In [ ]:
import pickle
from collections import Counter

import numpy as np
import pandas as pd

# Sanity check over PAS name to group map: they should be always the same

In [ ]:
anno_type = 'ageXclass'

In [ ]:
df_bedmeta_r = pd.read_table(f'/sc/arion/projects/CommonMind/yeon/p/APA/run_SCAPTURE/4_quant_pas/RUSH/pl_{anno_type}/pas_anno_renamed.bedmeta',
                             dtype={0: 'category'}, header=None)
df_bedmeta_h = pd.read_table(f'/sc/arion/projects/CommonMind/yeon/p/APA/run_SCAPTURE/4_quant_pas/HBCC/pl_{anno_type}/pas_anno_renamed.bedmeta',
                             dtype={0: 'category'}, header=None)
df_bedmeta_m = pd.read_table(f'/sc/arion/projects/CommonMind/yeon/p/APA/run_SCAPTURE/4_quant_pas/MSSM/pl_{anno_type}/pas_anno_renamed.bedmeta',
                             dtype={0: 'category'}, header=None)

df_group_to_genes = pd.read_pickle(f'/sc/arion/projects/CommonMind/yeon/p/APA/run_SCAPTURE/3_merge_pas/pl_{anno_type}/PAS_group_to_all_overlapped_genes.pkl')
                             

In [ ]:
df_bedmeta_m.head()

In [ ]:
df_group_to_genes.head()

In [ ]:
print((df_bedmeta_r == df_bedmeta_m).all().all())
print((df_bedmeta_r == df_bedmeta_h).all().all())

# Read original bed

In [ ]:
df_bed = pd.read_csv(f'../../3_merge_pas/pl_{anno_type}/merged/all_passed_merged_PAS-{anno_type}.bed', sep='\t', header=None, dtype={0:'category'})
df_bed

# Add evaluation & loci info

In [ ]:
df_name_and_group = pd.read_pickle(f'../../3_merge_pas/pl_{anno_type}/Each_PAS_with_group_{anno_type}.pkl')
print(df_name_and_group['name'].iloc[0])
df_name_and_group

In [ ]:
df_eval = pd.read_pickle(f'../../2_filter_pas/pl_{anno_type}/filtered_pas/filtered_peaks_evaluated_intersected.pkl.gz')
df_eval

In [ ]:
df_eval['name'] = df_eval[3] + '|' + df_eval['pas_from']
df_eval['AnnoMore2'] = df_eval['# Anno (without GENCODE)'] >= 2
df_eval['AnnoMore1'] = df_eval['# Anno (without GENCODE)'] >= 1
df_eval['AnnoMore1_DeepPASS'] = df_eval['AnnoMore1'] & df_eval['DeepPASS']

In [ ]:
name_to_group = dict(zip(df_name_and_group['name'], df_name_and_group['group']))
df_eval['group'] = df_eval['name'].map(name_to_group).astype('Int64')
df_eval = df_eval[~df_eval['group'].isnull()].copy()
df_eval

In [ ]:
def choose_region(se):
    priorities = '3UTR CDS exon intron 3primeExtended 5UTR'.split()
    mapped_regions = set(se)
    for region in priorities:
        if region in mapped_regions:
            return region

df_group_eval = df_eval.groupby('group')[['GENCODE', 'AnnoMore2', 'AnnoMore1', 
                                          'AnnoMore1_DeepPASS']].any()
df_group_eval['pas_region'] = df_eval.groupby('group')['pas_type'].apply(choose_region)
df_group_eval

In [ ]:
df_bed['group'] = df_bed[3].map(lambda x: x.split('|')[1]).astype('Int64')
df_bed_eval = pd.merge(df_bed, df_group_eval, left_on='group', right_index=True, how='left')
df_bed_eval

# Load all PAS-mapped genes

In [ ]:
df_pas_to_gene = pd.read_pickle(f'../../3_merge_pas/pl_{anno_type}/PAS_group_to_all_overlapped_genes.pkl')
group_to_genes = dict(zip(df_pas_to_gene['group'],  df_pas_to_gene['geneNameList']))
df_pas_to_gene

In [ ]:
df_bed_eval['mapped_genes'] = df_bed_eval['group'].map(group_to_genes).apply(lambda x: ';'.join(x))
df_bed_eval

# Add PAS names in SCAPTURE quant

In [ ]:
group_to_quantname = dict(zip(df_bedmeta_r[13], df_bedmeta_r[3]))
print(len(group_to_quantname) == len(df_bedmeta_r))

df_bed_eval['PAS_name'] = df_bed_eval['group'].map(group_to_quantname)
df_bed_eval.head()

# Add subregional info

## Load Gene GTF

In [ ]:
ensembl_gtf_path = '/sc/arion/projects/CommonMind/yeon/p/APA/ref/ensembl.104/Homo_sapiens.GRCh38.104.filt_renamed.gtf.gz'

# For fixed second_delim (delim is ' "' between gene_id and info: gene_id "ENSG00000284662")
def parse_desc(desc):
    return dict(field.strip().removesuffix('"').split(' "') for field in desc.removesuffix(';').split(';'))
	
df_gtf = pd.read_table(ensembl_gtf_path, header=None, comment='#', dtype={0: 'category'})
df_gtf_desc = pd.DataFrame(list(df_gtf[8].apply(parse_desc)),
                           index=df_gtf.index).convert_dtypes()

for col in df_gtf_desc:
    df_gtf[col] = df_gtf_desc[col]

In [ ]:
df_gtf

In [ ]:
gb_all = df_gtf.groupby('gene_name')
gb_Cds = df_gtf[df_gtf[2]=='CDS'].groupby('gene_name')

df_3utr = pd.DataFrame()
df_3utr['CDS_left'] = gb_Cds[3].min()
df_3utr['CDS_right'] = gb_Cds[4].max()
df_3utr['strand'] = gb_Cds[6].first()
df_3utr['gene_left'] = gb_all[3].min()
df_3utr['gene_right'] = gb_all[4].max()
df_3utr['gene_strand'] = gb_all[6].first()

df_3utr

In [ ]:
if (df_3utr['strand']==df_3utr['gene_strand']).all():
    del df_3utr['gene_strand']

In [ ]:
# GTF use one-based, closed coordiate system. So add or subtract 1 from CDS position. 
# np.where(condition, if true, if false)

df_3utr['3UTR_left'] = np.where(
    df_3utr['strand'] == '+', 
    df_3utr['CDS_right'] + 1, 
    df_3utr['gene_left']
)

df_3utr['3UTR_right'] = np.where(
    df_3utr['strand'] == '+', 
    df_3utr['gene_right'], 
    df_3utr['CDS_left'] - 1
)
df_3utr['3UTR_middle'] = (df_3utr['3UTR_left'] + df_3utr['3UTR_right']) / 2
df_3utr

## Define exon

In [ ]:
df_gtf_exon = df_gtf[df_gtf[2]=='exon'][['gene_name', 0, 3, 4, 6]].copy()

# Subclassify the region, by the primary gene name

In [ ]:
# Convert 0-based bed to 1-based, to be compared with GTF
df_bed_eval['start_1based'] = df_bed_eval[1] + 1
df_bed_eval['gene_name'] = df_bed_eval[3].apply(lambda x: x.split('|')[0])
df_bed_eval.head()

In [ ]:
def subclassify_3utr(se_pas, df_3utr):
    if (se_pas['pas_region'] != '3UTR'):
        return se_pas['pas_region']
    gn = se_pas['gene_name']
    
    try:
        se_3utr = df_3utr.loc[gn]
    except KeyError:
        print(f'{gn} seems non-coding, but SCAPTURE reported 3UTR! Maybe overlaped position with other gene')
        print(se_pas['mapped_genes'])
        print()
        df_exons = df_gtf_exon[df_gtf_exon.gene_name==gn].copy()
        # If PAS overlaps with exon, non-coding exon. otherwise intron
        se_exon_starts = df_exons[3]
        se_exon_ends = df_exons[4]
        if se_pas[5]=='+':
            pas_5p = se_pas['start_1based']
            pas_3p = se_pas[2]
        else:
            pas_5p = se_pas[2]
            pas_3p = se_pas['start_1based']
        start_within_exon = ((se_exon_starts <= pas_5p) & (pas_5p <= se_exon_ends)).any()
        end_within_exon = ((se_exon_starts <= pas_3p) & (pas_3p <= se_exon_ends)).any()
        if start_within_exon or end_within_exon:
            return 'exon'
        else:
            return 'intron'
    
    strand = se_pas[5]
    if strand=='+':
        if se_pas[2] < se_3utr['3UTR_left']:
            return 'Upstream_3UTR'
        elif se_pas[2] <= se_3utr['3UTR_middle']: # if exactly middle, proximal
            return 'Proximal_3UTR'
        # Not precise, but if any portion overlap with known 3UTR, just set as distal
        elif se_pas['start_1based'] <= se_3utr['3UTR_right']:
            return 'Distal_3UTR'
        else:
            print(se_pas['PAS_name'], 'is actually downstream!')
            return '3p Downstream'
    else: 
        if se_pas['start_1based'] > se_3utr['3UTR_right']:
            return 'Upstream_3UTR'
        elif se_pas['start_1based'] > se_3utr['3UTR_middle']:
            return 'Proximal_3UTR'
        # Not precise, but if any portion overlap with known 3UTR, just set as distal
        elif se_pas[2] >= se_3utr['3UTR_left']:
            return 'Distal_3UTR'
        else:
            print(se_pas['PAS_name'], 'is actually downstream!')
            return '3p Downstream'
        
df_bed_eval['pas_subclassified'] = df_bed_eval.apply(subclassify_3utr, args=[df_3utr], axis=1)

In [ ]:
# Check how many are 3UTR -> not 3UTR
Counter(df_bed_eval[(df_bed_eval.pas_region=='3UTR')]['pas_subclassified'])

In [ ]:
Counter(df_bed_eval.pas_subclassified)

In [ ]:
Counter(df_bed_eval.pas_region)

In [ ]:
# Save these to check distal and proximal 3UTR list by IGV
df_bed_eval[df_bed_eval['pas_subclassified']=='Distal_3UTR'][range(12)].to_csv('distal_3utr_for_igv.bed', sep='\t', header=None, index=False)
df_bed_eval[df_bed_eval['pas_subclassified']=='Proximal_3UTR'][range(12)].to_csv('proximal_3utr_for_igv.bed', sep='\t', header=None, index=False)

# Define visualized name

## Define 3p end order

In [ ]:
def order_from_rna_5p(df):
    strand = df[5].iloc[0]
    if strand == '+':
        # If +, rank of end. if tie, use start (by first)
        df['order'] = df.sort_values(1)[2].rank(method='first').astype(int)
        return df[['PAS_name', 'order']]
    elif strand == '-':
        # If -, reverse rank of start. if tie, use end
        df['order'] = df.sort_values(2, ascending=False)[1].rank(ascending=False, method='first').astype(int)
        return df[['PAS_name', 'order']]
    else:
        raise ValueError('Strand must be + or -, following given: ' + strand)

df_pas_order_in_gene = df_bed_eval.groupby('gene_name').apply(order_from_rna_5p)
df_pas_order_in_gene

In [ ]:
df_pas_order_in_gene = df_pas_order_in_gene.reset_index().set_index('level_1').sort_index()

In [ ]:
print(all(df_bed_eval['gene_name'] == df_pas_order_in_gene['gene_name']))
print(all(df_bed_eval['PAS_name'] == df_pas_order_in_gene['PAS_name']))
df_bed_eval['PAS_index_from5p'] = df_pas_order_in_gene['order']
df_bed_eval

## Check renamed PAS index results

In [ ]:
se_pas_index = df_bed_eval['PAS_name'].apply(lambda x: x.split('-')[-1]).astype(int)
se_pas_cnt_per_gene = df_bed_eval.groupby('gene_name')['PAS_name'].count()

### If single PAS, all should be 1

In [ ]:
df_bed_eval_single_pas = df_bed_eval[df_bed_eval['gene_name'].isin(se_pas_cnt_per_gene[se_pas_cnt_per_gene==1].index)]
set(df_bed_eval_single_pas['PAS_index_from5p'])

### If + strand, they should be the same

In [ ]:
df_bed_plus = df_bed_eval[df_bed_eval[5]=='+']
(df_bed_plus['PAS_name'].apply(lambda x: int(x.split('-')[-1])) == df_bed_plus['PAS_index_from5p']).all()

In [ ]:
df_pas_order_in_gene.loc[[6, 7, 8, 9, 10]]

In [ ]:
df_bed_plus[df_bed_plus['PAS_name'].apply(lambda x: int(x.split('-')[-1])) !=df_bed_plus['PAS_index_from5p'] ]

### If - strand, their should be 1 to number of PAS

In [ ]:
df_bed_minus = df_bed_eval[df_bed_eval[5]=='-']
(df_bed_minus.groupby('gene_name')['PAS_index_from5p'].max() -  df_bed_minus.groupby('gene_name')['PAS_name'].count()).describe()

## Define region

In [ ]:
Counter(df_bed_eval['pas_subclassified'])

In [ ]:
Counter(df_bed_eval['pas_region'])

In [ ]:
shortnames = {
    'intron': 'int',
    '3UTR': '3ut',
    'exon': 'nce',
    'CDS': 'cds',
    '3primeExtended': '3ds',
    '5UTR': '5ut',
}

In [ ]:
df_bed_eval['visual_name'] = df_bed_eval['gene_name'] + ':' + df_bed_eval['PAS_index_from5p'].astype(str) + '_' + df_bed_eval['pas_region'].map(shortnames)
df_bed_eval['visual_name']

### Save a bed file with visual names

In [ ]:
df_bed_eval[[0, 1, 2, 'visual_name', 4, 5, 6, 7, 8, 9, 10, 11]].to_csv(f'PAS_merged_evaluated_with_visual_names_{anno_type}.bed', sep='\t', header=None, index=None)

# Save the final table

In [ ]:
del df_bed_eval['start_1based']
df_bed_eval

In [ ]:
df_bed_eval.to_csv(f'PAS_merged_evaluated_with_name_{anno_type}.tsv', sep='\t')